# Questions for Nick

- hospitalization_id or stitched?
- vent needs to be in icu

# Plan

## Objective

Estimate, for each day of a 28-day window (Day 1 → Day 28), the number of eligible patients and the summary statistics of VFD-28 (ventilator-free days at 28 days) among mechanically ventilated patients — stratified by ICU type and by the year/ICU-type-specific pool of eligible providers — to support power calculations for a study of provider-level practice variation in mechanical ventilation management.

**Reference for VFD-28 definition & statistical guidance:** Yehya N, et al. *Reappraisal of Ventilator-Free Days in Critical Care Research.* Am J Respir Crit Care Med. 2019. https://pmc.ncbi.nlm.nih.gov/articles/PMC6812447/

## Cohort Eligibility (index criteria, evaluated only at the moment MV starts)

**Inclusion**
- Adults, `age_at_admission` ≥ 18
- Invasive mechanical ventilation (`device_category == IMV` in `respiratory_support`) **while physically located in an ICU** (`location_category == icu` in `adt`) — MV started outside the ICU (e.g., in the ED) doesn't count until/unless it continues into an ICU-located record.
- First **ICU** MV episode of the hospitalization only — if extubated and later reintubated, only the first ICU episode counts. **Day 1 is always the first day of that first ICU episode.** A patient ventilated only outside the ICU (never on IMV while in an ICU location) is excluded from the cohort entirely, not just delayed to a later Day 1.

**Exclusion (baked into the cohort — evaluated *at MV onset only*, never looking forward)**
- **ECMO at MV onset** (not ECMO at any point during the hospitalization). A patient cannulated onto ECMO on day 3 stays in — their Day-1 provider assignment was already legitimate when the day-1 trial happened.
- **Tracheostomy already in place at MV onset** (not tracheostomy placed later). Operationally: a trach documented within the first 24 hours **of the first ICU IMV record** (`mv_start_dttm`) is assumed to have been in place at onset — anchored to the ICU-based Day 1, not the patient's true first-ever intubation moment if that happened earlier outside the ICU.

**Flag columns only — NOT baked into the cohort exclusion.** Return as separate flags so the sample-size cost of each can be evaluated before committing to excluding them:
- **Cardiac arrest / anoxic brain injury, present on admission** — from `hospital_diagnosis`:
  - `flag_cardiac_arrest` — ICD-10 `I46.x`, only where `poa_present == 1`
  - `flag_anoxic_injury` — ICD-10 `G93.1`, only where `poa_present == 1`
  - Also return `diagnosis_primary` for comparison. POA-only is deliberate — an arrest coded without the POA flag could have happened on day 5 (after ventilation started) and should not exclude anyone.
- **Do-not-intubate status before MV initiation** — from `code_status`: take the status with the latest `start_dttm` at or before MV start.
  - `flag_dni` = 1 only for the three DNI-containing categories: **DNR/DNI, DNAR/DNI, DNI_only**
  - Plain **DNR, DNAR, and UDNR stay in** (not flagged) — in CLIF, DNR means "no CPR but do intubate," so these patients wanted the ventilator and belong in the study.

## VFD-28 Definition (Yehya et al. 2019 standard, with one deliberate deviation)

Computed once per patient's first vent episode, anchored to a fixed 28-day window starting Day 1 (vent initiation):

- **VFD-28 = 0** if the patient dies within 28 days of vent initiation (regardless of ventilation status at death)
- **VFD-28 = 0** if the patient is still on IMV at Day 28, or if no final extubation is ever observed before censoring
- **VFD-28 = 28 − x** if liberated from IMV on day x (extubated and off positive-pressure ventilation, with no subsequent reintubation observed before censoring)
- **Reintubation:** if reintubated within 28 days, VFD-28 is counted from the day of the *final* extubation, not the first
- **Deaths after Day 28 are censored** (ignored) — only 28-day status matters
- **No minimum sustained-liberation duration is required** (deliberate deviation from the article, decided 2026-08-19): Yehya et al. recommend requiring >48 continuous hours off support before crediting a "successful extubation," but that standard is framed specifically around avoiding credit for patients who get *reintubated* — it isn't about patients who are discharged or otherwise censored shortly after a clean extubation, which the article doesn't address at all. Any final, non-reintubated extubation now counts immediately, however briefly it was observed before the censor point (discharge or Day 28) — e.g., a patient extubated on Day 20 and discharged home 6 hours later gets VFD28 = 8, not 0.
- **Discharge (placeholder, to revisit):** for now, censor at discharge the same way we censor at death/day-28 — i.e., no further ventilation status is tracked past discharge; VFD-28 is locked in based on in-hospital data as of the discharge date. This is a simplifying assumption pending further discussion, not a final decision.

**Day-anchoring (confirmed by Jared):** VFD-28 does **not** get recomputed for each day of the loop. Each patient has exactly one VFD-28 value, fixed relative to their own Day 1 — only the *set* of at-risk patients changes as the loop advances through Day 1 → Day 28.

**Required summary statistics per stratum:** mean, **SD** (this is the number the whole power calculation turns on — prioritize getting this right), median, IQR, proportion at 0, and proportion at 28.


## Provider Eligibility & ICU-type Stratification

A provider (`ProvID`) is **eligible** for a given ICU type/year if they practiced (were active) in that `icu_type` during that year. Eligibility is nested by unit — a medical ICU patient could only ever have been covered by a medical ICU provider — so trials are stratified by (ICU type, year) and the counts must match that stratification. (No role/title field is available in the real provider data — "eligible provider" means any provider, not specifically attendings; see QC notes.)

**Two versions of "number of eligible providers," both needed:**

**(a) Per patient** — for each at-risk patient on each day, the count of eligible providers they *could* have been assigned to that day (i.e., all providers active that year in that patient's `icu_type`). Report the **distribution** across patients — median and IQR is sufficient, no need for the raw list per patient.

**(b) Per ICU per year (roster table)**:

| year | hospital_id |icu_type    | eligible_providers (ProvID list) | N |
|------|--|-----------|------------------------------------|---|
| 2011 | ref | medical     | […]                                 | … |
| 2011 | 1 |surgical    | […]                                 | … |
| 2011 | 2 |neuro       | […]                                 | … |
| 2011 | 2 |cardiovascular | […]                              | … |
| …    | |…           | …                                    | … |

**Stratification scope:**
- **East Bank only:** break out by ICU type **and** year — medical, surgical, neuro, cardiovascular.
- **Community hospitals:** do **not** split by unit type — the same provider roster covers all their units, so report at the hospital level only (still by year).

Jared's rule of thumb governing all of this: the sample sizes actually used in the analysis are the ones we power on — so patient counts and provider counts must be reported at the same stratification granularity that will be used in the eventual trial-level analysis.


## Daily Landmark Workflow

For each Day *d* = 1, 2, …, 28, and separately for each stratum (East Bank: hospital × ICU type × year; community hospitals: hospital × year):

1. **Identify patients at risk on Day *d*** — a patient counts if, **at the start of Day *d***, they are:
   - Still alive
   - Still on the ventilator
   - Not yet discharged (censored at discharge, same treatment as death — see VFD-28 Definition)
   - Still in the unit (`icu_type`)
   - Covered by an eligible provider (`ProvID` active that year in that `icu_type`)
   - A member of the base cohort (adult, first MV episode, no ECMO at onset, no trach at onset)
2. **Count:**
   - N patients at risk (Day *d*, stratum)
   - N eligible providers per patient (Day *d*, stratum) — distribution (median, IQR) across the at-risk patients
   - N eligible providers per ICU/year — roster count from the lookup table (East Bank only; hospital-level for community sites)
3. **Compute VFD-28 summary statistics** for the Day-*d* at-risk patient set:
   - Mean, **SD**, median, IQR
   - Proportion at 0
   - Proportion at 28
4. Repeat steps 1–3 for every day 1–28, every stratum in scope.

This produces a series of 28 "landmark" snapshots per stratum — the at-risk patient *set* shrinks day to day (patients drop out on liberation, death, discharge, or transfer out of unit), but each patient's own VFD-28 value never changes once computed. A patient who exits the risk set on Day *d* (e.g., liberated) still contributes their fixed VFD-28 value to every day's summary stats through Day *d*, just not to Day *d*+1 onward.

**Why "still on the ventilator" is part of at-risk, not just death/discharge:** the daily count isn't tracking the outcome (that's already fixed) — it's tracking who is still exposed to a provider's active ventilator-management decision that day. Once liberated, there's no more vent decision to attribute to a provider, so the patient exits the *denominator* even though their VFD-28 value keeps flowing into every day's stats up through their last at-risk day.


## Deliverables

- **Cohort table:** eligible patients with `flag_cardiac_arrest`, `flag_anoxic_injury` (+ `diagnosis_primary`), and `flag_dni` columns attached (not pre-excluded) so each exclusion's sample-size cost can be evaluated
- **Roster lookup table:** eligible providers by (year, ICU type) — East Bank; by (year, hospital) — community
- **Daily table:** N patients at risk, by (day 1–28) × stratum
- **Daily table:** N eligible providers per patient (median, IQR distribution) and N eligible providers per ICU/year (roster count), by (day 1–28) × stratum
- **Daily table:** VFD-28 summary stats (mean, SD, median, IQR, proportion at 0, proportion at 28) for the Day-*d* at-risk cohort, by (day 1–28) × stratum
- All outputs aggregate-only (cell counts ≥ 10) per CLIF federated-analysis policy — no patient-level export


## Open Questions

**Resolved (2026-08-19):**
- ~~Discharge before Day 28 undercounting liberation~~ — resolved by removing the 48h minimum-liberation-duration requirement entirely (see VFD-28 Definition). A final, non-reintubated extubation now counts regardless of how briefly it was observed before censoring, so this no longer undercounts patients discharged shortly after extubation.

**Deferred for now (placeholder decisions in place, revisit later):**
- **Discharge before Day 28, general handling** — *placeholder in place:* censor at discharge the same way we censor at death (no post-discharge follow-up assumed) — i.e., we don't know what happens after discharge, we just stop looking. Still a simplifying assumption, not a final decision.
- **Community hospital list & East Bank ICU-type list** — resolved via real data: `hospital_type` (`academic`=East Bank, `community`=community sites) and `location_type` (specific ICU subtype, populated only for ICU rows) are both confirmed and in use as of the real-data pipeline (Steps 3–4).


# Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import json
import time
import clifpy
import polars as pl
import duckdb

# Global Settings

In [ ]:
pd.set_option('display.max_columns', None)

os.makedirs('output_no_share', exist_ok=True)
os.makedirs('output_to_box', exist_ok=True)

con = duckdb.connect(database='output_no_share/cardiac_hospitalization_id.duckdb')

with open("config.json", "r", encoding="utf-8-sig") as f:
    cfg = json.load(f)

clif_path = cfg["data_directory"]
print(f"CLIF filepath: {clif_path}")

file_type = cfg['filetype']
print(f'Filetype: {file_type}')

time_zone = cfg['timezone']
print(f'Timezone: {time_zone}')

site_name = cfg.get('site_name', 'site')
print(f'Site name: {site_name}')

con.execute(f"SET TimeZone = '{time_zone}'")

# File paths

In [ ]:
ext = file_type   # "parquet" or "csv"
hosp_diagnosis_path    = f"{clif_path}/clif_hospital_diagnosis.{ext}"
procedure_path         = f"{clif_path}/clif_patient_procedures.{ext}"
hospitalization_path   = f"{clif_path}/clif_hospitalization.{ext}"
dnr_path               = f"{clif_path}/clif_code_status.{ext}"
patient_path           = f"{clif_path}/clif_patient.{ext}"
vent_path              = f"{clif_path}/clif_respiratory_support.{ext}"
dialysis_path          = f"{clif_path}/clif_crrt_therapy.{ext}"
patient_diagnosis_path = f"{clif_path}/clif_patient_diagnosis.{ext}"
adt_path               = f"{clif_path}/clif_adt.{ext}"
vitals_path            = f"{clif_path}/clif_vitals.{ext}"
assessment_path        = f"{clif_path}/clif_patient_assessments.{ext}"
micro_culture_path     = f"{clif_path}/clif_microbiology_culture.{ext}"
labs_path              = f"{clif_path}/clif_labs.{ext}"
intermittent_med_path  = f"{clif_path}/clif_medication_admin_intermittent.{ext}"
continuous_med_path    = f"{clif_path}/clif_medication_admin_continuous.{ext}"
provider_path          = f"{clif_path}/clif_provider.{ext}"
ecmo_path              = f"{clif_path}/clif_ecmo_mcs.{ext}"

In [ ]:
# REQUIRED = {
#     hosp_diagnosis_path:  ["hospitalization_id", "diagnosis_code", "poa_present", "diagnosis_primary"],
#     procedure_path:       ["hospitalization_id", "procedure_code", "procedure_billed_dttm"],
#     hospitalization_path: ["hospitalization_id", "patient_id", "admission_dttm", "discharge_dttm",
#                            "age_at_admission", "admission_type_name", "admission_type_category",
#                            "discharge_category"],
#     adt_path:             ["hospitalization_id", "hospital_id", "in_dttm", "out_dttm", "location_category"],
#     patient_path:         ["patient_id", "sex_category", "race_category", "ethnicity_category",
#                            "language_category", "death_dttm"],
#     assessment_path:      ["hospitalization_id", "recorded_dttm", "assessment_category",
#                            "numerical_value"],
#     vent_path:            ["hospitalization_id", "recorded_dttm", "device_category", "tracheostomy"],
#     dnr_path:              ["patient_id", "start_dttm", "code_status_category"],
#     provider_path:         ["hospitalization_id", "provider_id", "start_dttm", "stop_dttm", "provider_role_category"],
#     ecmo_path:              ["hospitalization_id", "recorded_dttm"],
# }

# errors = []
# for path, required_cols in REQUIRED.items():
#     name = os.path.basename(path)
#     if not os.path.exists(path):
#         errors.append(f"  ✗ MISSING FILE:  {name}")
#         continue
#     actual_cols = set(duckdb.sql(f"SELECT * FROM '{path}' LIMIT 0").columns)
#     missing = [c for c in required_cols if c not in actual_cols]
#     if missing:
#         errors.append(f"  ✗ MISSING COLS:  {name} → {missing}")
#     else:
#         print(f"  ✓ {name}")

# if errors:
#     print("\nPRE-FLIGHT FAILED:")
#     for e in errors:
#         print(e)
#     raise RuntimeError("Fix the above issues before running the notebook.")
# else:
#     print("\nAll required CLIF tables present and columns verified. Ready to run.")

# Pipeline Constants

All tunable rules from the Plan are centralized here — change once, applies everywhere below.

In [ ]:
ECMO_ONSET_BUFFER_HOURS  = 0     # ECMO recorded at/before MV start (+buffer) => "on ECMO at onset" (excluded).
                                  # Kept at 0 to strictly honor "evaluated only at MV onset, never looking
                                  # forward" — a >0 buffer would exclude patients cannulated onto ECMO shortly
                                  # AFTER MV started, which the eligibility rule explicitly says must stay in.
TRACH_ONSET_WINDOW_HOURS = 24    # tracheostomy flag within this window of MV start => "already in place at onset" (excluded)
VFD_WINDOW_DAYS          = 28

# device_category values that represent being OFF positive-pressure support outright (liberated),
# regardless of tracheostomy status. Stored lower-case to match the lower()/LOWER() comparisons
# used everywhere this is checked (Steps 2b, 4a) — device_category casing isn't guaranteed
# consistent across sites/extracts.
OFF_SUPPORT_DEVICES = ("room air", "nasal cannula", "face mask", "trach collar")

# Noninvasive support devices — per Yehya et al. 2019, these should NOT be held against liberation
# for a standard (non-tracheostomized) patient ("we recommend not counting noninvasive support"
# toward VFD; a patient extubated then bridged to NIV/HFNC is still credited as liberated from IMV).
# The stricter "off ALL positive-pressure support" standard is scoped by the article specifically to
# TRACHEOSTOMIZED patients ("tracheostomies should be treated as other invasive ventilation").
# So: at a given respiratory_support row, NIV_DEVICES count as OFF unless tracheostomy=1 at that row
# (i.e., the patient already has a trach in place, in which case NIV/HFNC use is treated as still-on
# support, same as IMV). IMV itself is always ON. Any other/unrecognized device_category is ON
# (conservative default). Stored lower-case, same convention as OFF_SUPPORT_DEVICES.
NIV_DEVICES = ("nippv", "high flow nc")

# Step 1 — Cohort Identification

Day 1 = the first ICU IMV record for a hospitalization (the earliest respiratory_support row where the patient is simultaneously on IMV and physically located in an ICU — found via an ASOF join to the nearest-prior ADT row, then filtered to `location_category = 'icu'`). MV that starts outside the ICU (e.g., in the ED) doesn't establish Day 1 by itself; a patient who is ventilated only outside the ICU is excluded from the cohort entirely. ECMO-at-onset and tracheostomy-at-onset are evaluated only in a window around that (ICU-anchored) moment, never looking forward. Cardiac-arrest/anoxic-injury (POA) and DNI status are computed as flags, not exclusions.

In [ ]:
# --- 1a: Day 1 index — first-ever ICU IMV record per hospitalization ---
mv_first = con.execute(f"""
WITH all_vent_location AS (
    SELECT
        vp.hospitalization_id,
        vp.recorded_dttm,
        adt.in_dttm,
        adt.location_category
    FROM '{vent_path}' vp
    ASOF INNER JOIN '{adt_path}' adt
        ON vp.hospitalization_id = adt.hospitalization_id
        AND vp.recorded_dttm >= adt.in_dttm
    WHERE LOWER(vp.device_category) = 'imv'
        AND LOWER(adt.location_category) = 'icu'
)
SELECT
    hospitalization_id,
    MIN(recorded_dttm) AS mv_start_dttm
FROM all_vent_location
GROUP BY hospitalization_id
""").df()
con.register("mv_first", mv_first)
print(f"Hospitalizations with >=1 ICU IMV record: {len(mv_first):,}")

In [ ]:
# --- 1b: base cohort — join hospitalization + patient, apply age filter ---
cohort_base = con.execute(f"""
    SELECT
        h.hospitalization_id,
        h.patient_id,
        m.mv_start_dttm,
        h.age_at_admission,
        h.admission_dttm,
        h.discharge_dttm,
        h.discharge_category,
        pt.death_dttm
    FROM mv_first m
    JOIN '{hospitalization_path}' h USING (hospitalization_id)
    JOIN '{patient_path}' pt USING (patient_id)
    WHERE h.age_at_admission >= 18
""").df()
con.register("cohort_base", cohort_base)
print(f"After age >= 18 filter: {len(cohort_base):,}")

In [ ]:
# --- 1c: ECMO-at-onset exclusion ---
ecmo_onset_ids = set(con.execute(f"""
    SELECT b.hospitalization_id
    FROM cohort_base b
    JOIN '{ecmo_path}' e USING (hospitalization_id)
    WHERE LOWER(e.device_category) = 'va_ecmo'
    GROUP BY b.hospitalization_id, b.mv_start_dttm
    HAVING MIN(e.recorded_dttm) <= b.mv_start_dttm + INTERVAL '{ECMO_ONSET_BUFFER_HOURS} hours'
""").df()["hospitalization_id"])
print(f"ECMO-at-onset excluded: {len(ecmo_onset_ids):,}")

In [ ]:
# --- 1d: tracheostomy-at-onset exclusion ---
trach_onset_ids = set(con.execute(f"""
    SELECT DISTINCT b.hospitalization_id
    FROM cohort_base b
    JOIN '{vent_path}' r USING (hospitalization_id)
    WHERE r.tracheostomy = 1
      AND r.recorded_dttm BETWEEN b.mv_start_dttm AND b.mv_start_dttm + INTERVAL '{TRACH_ONSET_WINDOW_HOURS} hours'
""").df()["hospitalization_id"])
print(f"Tracheostomy-at-onset excluded: {len(trach_onset_ids):,}")

In [ ]:
excluded_ids = ecmo_onset_ids | trach_onset_ids
cohort = cohort_base[~cohort_base["hospitalization_id"].isin(excluded_ids)].copy()
con.register("cohort_ids", cohort[["hospitalization_id", "patient_id", "mv_start_dttm", "admission_dttm"]])
print(f"Base cohort after ECMO/trach-at-onset exclusions: {len(cohort):,} "
      f"(excluded {len(excluded_ids):,} of {len(cohort_base):,})")

In [ ]:
# --- 1e: flag columns (NOT exclusions) — POA cardiac arrest / anoxic injury, DNI status ---
# diagnosis_code formatting is inconsistent (some rows have the decimal, e.g. "I46.2",
# others don't, e.g. "G931") — REPLACE(...,'.','') normalizes both before matching.
dx_flags = con.execute(f"""
    SELECT hospitalization_id,
           MAX(CASE WHEN REPLACE(diagnosis_code, '.', '') ILIKE 'I46%' AND poa_present = 1
                     THEN 1 ELSE 0 END) AS flag_cardiac_arrest,
           MAX(CASE WHEN REPLACE(diagnosis_code, '.', '') ILIKE 'I46%' AND poa_present = 1 AND diagnosis_primary = 1
                     THEN 1 ELSE 0 END) AS flag_cardiac_arrest_primary,
           MAX(CASE WHEN REPLACE(diagnosis_code, '.', '') ILIKE 'G931%' AND poa_present = 1
                     THEN 1 ELSE 0 END) AS flag_anoxic_injury,
           MAX(CASE WHEN REPLACE(diagnosis_code, '.', '') ILIKE 'G931%' AND poa_present = 1 AND diagnosis_primary = 1
                     THEN 1 ELSE 0 END) AS flag_anoxic_injury_primary
    FROM '{hosp_diagnosis_path}'
    WHERE hospitalization_id IN (SELECT hospitalization_id FROM cohort_ids)
    GROUP BY hospitalization_id
""").df()

In [ ]:
# code_status is patient-level: take the LATEST status at/before MV start (not "any DNI-ish status
# ever recorded between admission and MV start" — a patient who was DNR/DNI on admission but
# upgraded to Full before MV started should NOT be flagged). Ranked per-hospitalization (not
# per-patient) so a patient with multiple hospitalizations gets each one's own correct cutoff.
# code_status_category ILIKE '%dni%' (substring, case-insensitive) — not an exact match against a
# fixed category list. Functionally equivalent to exact-matching DNR/DNI, DNAR/DNI, DNI_only given
# the known category universe (Full, DNR, DNAR, UDNR, AND, DNR/DNI, DNAR/DNI, DNI_only — none of
# the "stay in the study" categories contain "dni" as a substring), but would need revisiting if a
# new code_status_category ever appeared containing "dni" without actually being a DNI status.
# No lower bound on start_dttm: "at or before MV start" per spec, not "since admission" — a
# standing pre-admission DNI order should still count.
dni_flags = con.sql(f"""
    WITH ranked AS (
        SELECT c.hospitalization_id, dni.code_status_category,
               ROW_NUMBER() OVER (PARTITION BY c.hospitalization_id ORDER BY dni.start_dttm DESC) AS rn
        FROM cohort_ids c
        INNER JOIN '{dnr_path}' dni USING (patient_id)
        WHERE dni.start_dttm <= c.mv_start_dttm
    )
    SELECT hospitalization_id,
           CASE WHEN code_status_category ILIKE '%dni%' THEN 1 ELSE 0 END AS flag_dni,
           code_status_category AS dni_source_category
    FROM ranked WHERE rn = 1
""").df()

In [ ]:
cohort = cohort.merge(dx_flags, on="hospitalization_id", how="left")
cohort = cohort.merge(dni_flags, on="hospitalization_id", how="left")
for c in ["flag_cardiac_arrest", "flag_cardiac_arrest_primary", "flag_anoxic_injury", "flag_anoxic_injury_primary", "flag_dni"]:
    cohort[c] = cohort[c].fillna(0).astype(int)

print(f"flag_cardiac_arrest=1: {cohort['flag_cardiac_arrest'].sum():,}  "
      f"flag_anoxic_injury=1: {cohort['flag_anoxic_injury'].sum():,}  "
      f"flag_dni=1: {cohort['flag_dni'].sum():,}  ")
cohort

# Step 2 — VFD-28 Computation

Follows Yehya et al. 2019: `x` = number of **full days elapsed** since MV start (Day 0 = initiation), `VFD28 = 28 - x`. This is why x uses `floor(elapsed_hours / 24)`, not a 1-indexed day label — using the 1-indexed "Day d" label here would make VFD28=28 mathematically unreachable (validated during testing: the off-by-one was caught by checking `proportion at 28` against synthetic data and finding it was always 0).

Reintubation is handled by walking the full device-transition sequence per patient and keeping only the **last** off-support period that has no subsequent reintubation before censoring — matching "counted from the day of final extubation." No minimum sustained-liberation duration is required (see VFD-28 Definition above for why the 48h rule was dropped) — any final, non-reintubated extubation counts immediately, however briefly it was observed before censoring.

Discharge censoring (placeholder decision) falls out naturally here: `censor_dttm = min(mv_start + 28 days, discharge_dttm)`, and the device-transition walk simply never sees data past that point.

In [ ]:
# --- 2a: window / censor timestamps ---
cohort["window_end_dttm"] = cohort["mv_start_dttm"] + pd.Timedelta(days=VFD_WINDOW_DAYS)
cohort["died_in_window"] = cohort["death_dttm"].notna() & (cohort["death_dttm"] <= cohort["window_end_dttm"])
cohort["censor_dttm"] = cohort[["window_end_dttm", "discharge_dttm"]].min(axis=1)
con.register("cohort_censor", cohort[["hospitalization_id", "mv_start_dttm", "censor_dttm"]])

def elapsed_full_days(mv_start, t):
    """x = number of full calendar days elapsed since initiation (Day 0 = initiation)."""
    return int(np.floor((t - mv_start).total_seconds() / 86400))

# Day-label cutoffs, computed with the SAME floor-day accounting VFD-28 itself uses below, so
# Step 4's at-risk loop and this step's censoring agree on which "Day d" a discharge/death instant
# falls into. (Without this, comparing raw timestamps independently in each step with different
# </= boundary conventions silently drops patients from the at-risk set one day early whenever
# their discharge/death lands exactly on a 24h-multiple offset from mv_start_dttm.)
# Vectorized (not .apply(axis=1)) — same floor-day math as elapsed_full_days above, verified
# equivalent, but matters for performance on a real (much larger) cohort.
cohort["last_trackable_day"] = ((cohort["censor_dttm"] - cohort["mv_start_dttm"]).dt.total_seconds() // 86400) + 1
cohort["death_day"] = ((cohort["death_dttm"] - cohort["mv_start_dttm"]).dt.total_seconds() // 86400) + 1

# --- 2b: pull in-window respiratory_support records for the cohort ---
resp_window = con.execute(f"""
    SELECT r.hospitalization_id, r.recorded_dttm, r.device_category, r.tracheostomy
    FROM '{vent_path}' r
    JOIN cohort_censor c ON c.hospitalization_id = r.hospitalization_id
    WHERE r.recorded_dttm >= c.mv_start_dttm AND r.recorded_dttm <= c.censor_dttm
    ORDER BY r.hospitalization_id, r.recorded_dttm
""").df()
# NIV/HFNC only counts as ON (blocking liberation) for tracheostomized patients — per Yehya et al.
# 2019, the "off ALL positive-pressure support" standard is scoped specifically to tracheostomy;
# a standard extubated patient bridged to NIV/HFNC is still credited as liberated from IMV.
# .fillna(0) makes the missing-tracheostomy default explicit (treat as not-tracheostomized) —
# must match the COALESCE(...,0) used in Step 4's SQL version of this same check (cell 33), since
# both are evaluating the same underlying respiratory_support rows and need to agree.
# device_category matched case-insensitively via .str.lower() — OFF_SUPPORT_DEVICES/NIV_DEVICES
# are already stored lower-case (Pipeline Constants), so only this side needs lower()-ing.
device_lower = resp_window["device_category"].str.lower()
trach_known = resp_window["tracheostomy"].fillna(0)
resp_window["state"] = np.where(
    device_lower.isin(OFF_SUPPORT_DEVICES), "OFF",
    np.where(device_lower.isin(NIV_DEVICES) & (trach_known != 1), "OFF", "ON")
)
print(f"In-window respiratory_support records: {len(resp_window):,}")

In [ ]:
# --- 2c: final-liberation walk per patient ---
def find_final_liberation(group):
    """Walks the device-state sequence; returns the start of the LAST off-support period
    that has no subsequent reintubation before censoring — i.e., the final extubation, used
    to anchor VFD-28 regardless of how long it was actually observed before the censor point
    (discharge or Day 28). No minimum-duration confirmation is required: any final,
    non-reintubated off-period counts. (Previously required >=48 sustained off-support hours
    before crediting liberation, per Yehya et al.'s general "successful extubation" standard —
    removed because that standard is about avoiding credit for patients who get REINTUBATED,
    not about patients who are discharged/censored shortly after a clean extubation; the article
    doesn't address discharge timing at all, and a patient discharged alive shortly after
    extubation is, if anything, itself a clinical signal of stability, not evidence against it.)
    """
    off_start = None
    for _, row in group.sort_values("recorded_dttm").iterrows():
        if row["state"] == "OFF" and off_start is None:
            off_start = row["recorded_dttm"]
        elif row["state"] == "ON" and off_start is not None:
            off_start = None  # reintubated — this off-period doesn't count, keep looking
    return pd.Series({
        "liberated_dttm": off_start if off_start is not None else pd.NaT,
        "liberation_confirmed": off_start is not None,
    })

liberation = resp_window.groupby("hospitalization_id").apply(find_final_liberation).reset_index()
cohort = cohort.merge(liberation, on="hospitalization_id", how="left")
cohort["liberation_confirmed"] = cohort["liberation_confirmed"].fillna(False)
cohort

In [ ]:
# --- 2d: VFD-28 ---
def elapsed_full_days(mv_start, t):
    return int(np.floor((t - mv_start).total_seconds() / 86400))

def compute_vfd28(row):
    if row["died_in_window"]:
        return 0
    if row["liberation_confirmed"]:
        x = min(elapsed_full_days(row["mv_start_dttm"], row["liberated_dttm"]), VFD_WINDOW_DAYS)
        return max(0, VFD_WINDOW_DAYS - x)
    return 0  # still ventilated at censor — no final non-reintubated extubation observed

cohort["vfd28"] = cohort.apply(compute_vfd28, axis=1)

print(cohort["vfd28"].describe())
print(f"\nProportion VFD28=0:  {(cohort['vfd28'] == 0).mean():.3f}")
print(f"Proportion VFD28=28: {(cohort['vfd28'] == 28).mean():.3f}")
print(f"Died in window:      {cohort['died_in_window'].sum():,}")
print(f"Liberation confirmed:{cohort['liberation_confirmed'].sum():,}")

# Step 3 — Provider Roster (year × ICU type)

A provider is eligible for a given (hospital, icu_type, year) if their provider record overlapped an ICU-location ADT interval for that hospitalization, in that year. (No role/title field exists in the real provider data — every `prov_npi` counts, not specifically attendings; see QC notes.)

In [ ]:
# NEEDS TO BE A PROVIDER IN COHORT OF INTEREST
# 15 maybe 25 STARTS

In [ ]:
provider_roster = con.sql(f"""
WITH provider_hourly AS (
    -- one row per (hospitalization, hour, provider) observation
    SELECT
        hospitalization_id,
        CAST(recorded_date AS TIMESTAMP) + CAST(recorded_hour AS INTEGER) * INTERVAL '1 hour' AS recorded_dttm,
        prov_npi
    FROM '{provider_path}'
    WHERE prov_npi IS NOT NULL
), provider_hospital AS (
    -- ASOF = nearest-prior ADT row (current physical location at that hour), NOT an exact
    -- DATE+HOUR match against adt.in_dttm. The original version joined on exact (hospitalization_id,
    -- date, hour) equality with an INNER JOIN, which only kept a provider-hour row if it happened to
    -- fall on the EXACT hour of an ADT transfer-in timestamp — verified this drops ~75%+ of
    -- provider-hour rows even in a trivial 4-hour test case, since real ADT transfers happen only a
    -- handful of times per stay, not every hour. The filter on hospital_type/hospital_id is applied
    -- in the WHERE clause below, AFTER resolving the current location — putting it inside the ASOF
    -- ON clause would make the join skip past a genuinely-current non-matching row (e.g. an LTACH
    -- stop mid-stay) to find an older matching one, which would be wrong.
    SELECT
        ph.hospitalization_id,
        ph.recorded_dttm,
        ph.prov_npi,
        YEAR(ph.recorded_dttm) AS year,
        adt.hospital_id,
        adt.hospital_type,
        adt.location_type
    FROM provider_hourly ph
    ASOF LEFT JOIN '{adt_path}' adt
        ON ph.hospitalization_id = adt.hospitalization_id
       AND ph.recorded_dttm >= adt.in_dttm
    WHERE adt.hospital_type IN ('community', 'academic')
      AND adt.hospital_id != 'Missing'
), community_hospitals AS (
    SELECT
        year,
        hospital_id,
        ANY_VALUE(hospital_type) as hospital_type,
        NULL as location_type,
        LIST(distinct prov_npi) as prov_npi_list,
        COUNT(distinct prov_npi) as prov_npi_count
    FROM provider_hospital
    WHERE hospital_type = 'community'
    GROUP BY year, hospital_id
), academic_hospitals AS (
    SELECT
        year,
        hospital_id,
        ANY_VALUE(hospital_type) as hospital_type,
        location_type,
        LIST(distinct prov_npi) as prov_npi_list,
        COUNT(distinct prov_npi) as prov_npi_count
    FROM provider_hospital
    WHERE hospital_type = 'academic'
    GROUP BY year, hospital_id, location_type 
)
SELECT *
FROM community_hospitals
UNION ALL
SELECT *
FROM academic_hospitals

ORDER BY year, hospital_id, location_type
""").df()
provider_roster

# Step 4 — Daily At-Risk Landmark Loop (Day 1–28)

For each patient × each Day *d*, at-risk requires: alive, not yet discharged, still on the ventilator (real-time device state — see note below), still in an ICU location, and covered by a provider — evaluated at the **start** of Day *d* (`mv_start_dttm + (d-1) days`).

**Real-time state vs. VFD-28's "final liberation":** these are deliberately different computations. VFD-28 (Step 2) only cares about the *final* liberation. The at-risk "still on the ventilator" check needs the *actual* moment-to-moment device state — including brief off-periods that later get reintubated (which don't count toward VFD-28, but which do mean the patient is genuinely off the vent, and therefore not at-risk, for those particular days). DuckDB's `ASOF JOIN` finds the most recent device-state record at-or-before each day's start.

In [ ]:
# --- 4a: real-time device-state timeline (distinct from Step 2's "final liberation" logic) ---
# device_category matched case-insensitively via LOWER(), same convention as the pandas version in
# Step 2b — OFF_SUPPORT_DEVICES/NIV_DEVICES are already stored lower-case (Pipeline Constants).
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE resp_state AS
    SELECT r.hospitalization_id, r.recorded_dttm,
           CASE
               WHEN LOWER(r.device_category) IN {tuple(OFF_SUPPORT_DEVICES)} THEN 'OFF'
               -- COALESCE(...,0) matters: raw `r.tracheostomy != 1` is NULL (not TRUE) when
               -- tracheostomy is missing, so the CASE would silently fall through to ELSE 'ON'
               -- instead of 'OFF' — the opposite of how the pandas version (Step 2b) treats the
               -- exact same missing-value case (NaN != 1 evaluates True in pandas/numpy). Forcing
               -- both to default missing-tracheostomy to "not tracheostomized" keeps Step 2 and
               -- Step 4 in agreement for the same underlying respiratory_support row.
               WHEN LOWER(r.device_category) IN {tuple(NIV_DEVICES)} AND COALESCE(r.tracheostomy, 0) != 1 THEN 'OFF'
               ELSE 'ON'
           END AS state
    FROM '{vent_path}' r
    JOIN cohort_censor c ON c.hospitalization_id = r.hospitalization_id
    WHERE r.recorded_dttm >= c.mv_start_dttm AND r.recorded_dttm <= c.censor_dttm
""")

In [ ]:
# --- 4b: cohort x Day 1..28 scaffold ---
con.register("cohort_full", cohort)
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE days AS
    SELECT c.hospitalization_id, c.patient_id, c.mv_start_dttm, c.death_dttm,
           c.discharge_dttm, c.censor_dttm, c.vfd28, c.last_trackable_day, c.death_day,
           d.day,
           -- fixed-duration 24h steps (not calendar 'INTERVAL 1 day', which is DST-aware and can
           -- be 23 or 25 real hours across a transition) — keeps Day d's real-time instant aligned
           -- with Step 2's fixed pd.Timedelta(days=28) window accounting.
           c.mv_start_dttm + (d.day - 1) * INTERVAL '24 hours' AS day_start_dttm
    FROM cohort_full c
    CROSS JOIN generate_series(1, {VFD_WINDOW_DAYS}) AS d(day)
""")
print(f"Day-level rows (cohort x {VFD_WINDOW_DAYS} days): {con.execute('SELECT count(*) FROM days').fetchone()[0]:,}")

In [ ]:
# --- 4c: ASOF join for real-time on/off state at each day's start ---
con.execute("""
    CREATE OR REPLACE TEMP TABLE days_state AS
    SELECT d.*, COALESCE(rs.state, 'ON') AS state_at_day_start
    FROM days d
    ASOF LEFT JOIN resp_state rs
      ON d.hospitalization_id = rs.hospitalization_id
     AND d.day_start_dttm >= rs.recorded_dttm
""")

In [ ]:
# provider is sparse hourly-snapshot data (gaps are possible) — build a proper timestamp and use
# an ASOF (nearest-prior-hour) lookup, same reasoning as the ADT fix above: an exact DATE+HOUR
# match would silently drop has_provider on any hour with no recorded provider row, causing the
# same kind of collapse the ADT join had.
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE prov_hourly AS
    SELECT hospitalization_id,
           CAST(recorded_date AS TIMESTAMP) + CAST(recorded_hour AS INTEGER) * INTERVAL '1 hour' AS recorded_dttm,
           prov_npi
    FROM '{provider_path}'
    WHERE prov_npi IS NOT NULL
""")

daily = con.sql(f"""
    SELECT
        ds.hospitalization_id,
        ds.patient_id,
        ds.day,
        ds.day_start_dttm,
        ds.vfd28,
        adt.hospital_id,
        adt.hospital_type,
        adt.location_type,
        -- Roster/stratification key: per-unit for academic (East Bank) sites, pooled (NULL) for
        -- community sites — mirrors exactly how provider_roster (Step 3) is built (community CTE
        -- sets location_type=NULL), so the merge in 4f and the GROUP BY in Step 5 line up. Raw
        -- location_type is kept above too, for descriptive/QC purposes even on community rows.
        CASE WHEN adt.hospital_type = 'academic' THEN adt.location_type ELSE NULL END AS icu_stratum,
        YEAR(ds.mv_start_dttm) AS index_year,
        -- is_alive / not_discharged compare integer "Day d" labels (computed in Step 2 with the
        -- same floor-day accounting VFD-28 itself uses), not raw timestamps — keeps this boundary
        -- consistent with Step 2's censoring instead of independently re-deriving it here.
        (ds.death_day IS NULL OR ds.day <= ds.death_day) AS is_alive,
        (ds.day <= ds.last_trackable_day) AS not_discharged,
        (ds.state_at_day_start = 'ON') AS still_on_vent,
        -- location_type is NULL for every non-ICU location (ward/ED/etc. — confirmed against the
        -- real data: all non-ICU rows have location_type IS NULL) and populated only for ICU rows,
        -- so "location_type IS NOT NULL" is already exactly "currently in an ICU." hospital_id =
        -- 'Missing' (a low-N data-quality artifact — team confirmed: ignore project-wide) is
        -- excluded here, AFTER the ASOF join resolves the true current location, rather than by
        -- pre-filtering the ADT table before joining — pre-filtering would make the join skip a
        -- genuinely-current 'Missing' row and incorrectly carry forward a stale earlier location.
        (adt.location_type IS NOT NULL AND adt.hospital_id != 'Missing') AS still_in_unit,
        (pr.prov_npi IS NOT NULL) AS has_provider
    FROM days_state ds
    -- ASOF = "most recent ADT row at or before this hour" (nearest-prior lookup — confirmed this is
    -- exactly the intended semantics: even a row from the near future that's numerically "closer"
    -- is never considered, since the join condition structurally excludes anything after
    -- day_start_dttm). NOT an exact DATE+HOUR match: adt.in_dttm marks only the moment of transfer
    -- INTO a location; the patient stays there until their NEXT transfer (a later in_dttm), so the
    -- most recent transfer-in row at-or-before day_start_dttm is always their current location.
    ASOF LEFT JOIN '{adt_path}' adt
        ON ds.hospitalization_id = adt.hospitalization_id
       AND ds.day_start_dttm >= adt.in_dttm
    -- Same nearest-prior-hour logic for the (sparse) hourly provider snapshots.
    ASOF LEFT JOIN prov_hourly pr
        ON ds.hospitalization_id = pr.hospitalization_id
       AND ds.day_start_dttm >= pr.recorded_dttm
""").df()
daily

In [ ]:
daily["at_risk"] = (
    daily["is_alive"] & daily["not_discharged"] & daily["still_on_vent"]
    & daily["still_in_unit"].fillna(False) & daily["has_provider"].fillna(False)
)

# --- 4f: attach per-patient eligible-provider count from the roster (indexed at MV-start year) ---
# Merge on icu_stratum (not raw location_type) — provider_roster's community rows have
# location_type=NULL (pooled across units per Step 3), so merging on the raw per-patient
# location_type would never match a community-hospital patient and silently leave their
# eligible-provider count null. icu_stratum already mirrors that same pooling.
# provider_roster's count column is named prov_npi_count — renamed here to n_eligible_providers
# to match what Step 5 expects. Only the needed columns are selected to avoid duplicate/renamed
# hospital_type columns from colliding with daily's own hospital_type on the merge.
daily = daily.merge(
    provider_roster[["hospital_id", "location_type", "year", "prov_npi_count"]].rename(
        columns={"year": "index_year", "location_type": "icu_stratum", "prov_npi_count": "n_eligible_providers"}
    ),
    on=["hospital_id", "icu_stratum", "index_year"], how="left",
)

print(f"at_risk rows: {daily['at_risk'].sum():,} / {len(daily):,}")
daily.groupby("day")["at_risk"].sum()

# Step 5 — Daily Summary Aggregation & Output

One row per (day, hospital, icu_type, year), computed only over that day's at-risk patients. Cells with n_at_risk < 10 are flagged for suppression before anything leaves the site (CLIF federated policy).

In [ ]:
at_risk = daily[daily["at_risk"]].copy()
con.register("at_risk", at_risk)

daily_summary = con.execute("""
    SELECT
        day, hospital_id, icu_stratum, index_year AS year,
        COUNT(*) AS n_at_risk,
        AVG(vfd28) AS vfd28_mean,
        STDDEV(vfd28) AS vfd28_sd,
        MEDIAN(vfd28) AS vfd28_median,
        QUANTILE_CONT(vfd28, 0.25) AS vfd28_q1,
        QUANTILE_CONT(vfd28, 0.75) AS vfd28_q3,
        AVG(CASE WHEN vfd28 = 0 THEN 1.0 ELSE 0 END) AS vfd28_prop_0,
        AVG(CASE WHEN vfd28 = 28 THEN 1.0 ELSE 0 END) AS vfd28_prop_28,
        MEDIAN(n_eligible_providers) AS elig_providers_per_pt_median,
        QUANTILE_CONT(n_eligible_providers, 0.25) AS elig_providers_per_pt_q1,
        QUANTILE_CONT(n_eligible_providers, 0.75) AS elig_providers_per_pt_q3,
        MAX(n_eligible_providers) AS elig_providers_roster_n
    FROM at_risk
    GROUP BY day, hospital_id, icu_stratum, index_year
    ORDER BY day, hospital_id, icu_stratum, index_year
""").df()

daily_summary["suppressed_lt10"] = daily_summary["n_at_risk"] < 10
print(f"Stratum-days with n_at_risk < 10: {daily_summary['suppressed_lt10'].sum()} / {len(daily_summary)}")

# --- write outputs ---
cohort.drop(columns=["window_end_dttm"]).to_parquet("output_no_share/cohort.parquet", index=False)
provider_roster.to_parquet("output_to_box/provider_roster.parquet", index=False)
daily_summary[~daily_summary["suppressed_lt10"]].to_csv("output_to_box/daily_summary.csv", index=False)
daily_summary.to_parquet("output_no_share/daily_summary_unsuppressed.parquet", index=False)

print("\nWrote: output_no_share/cohort.parquet, output_no_share/daily_summary_unsuppressed.parquet")
print("Wrote: output_to_box/provider_roster.parquet, output_to_box/daily_summary.csv (cells <10 suppressed)")
daily_summary